# Bài 1: Ảnh dưới dạng mảng (Images as Arrays)

Ảnh số là một lưới 2D gồm các con số. Trong bài này, chúng ta tự tạo một ảnh nhỏ từ đầu bằng NumPy, tìm hiểu cách biểu diễn ảnh xám (grayscale) và ảnh màu, rồi hiển thị chúng bằng Matplotlib.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['rose.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Ảnh xám là một mảng 2D

Mỗi phần tử là cường độ (intensity) của một điểm ảnh (pixel). Với ảnh 8-bit, giá trị điểm ảnh nằm trong khoảng từ 0 (đen) đến 255 (trắng).

In [ ]:
im = np.zeros((100, 100), dtype=np.uint8)
im[20:80, 20:80] = 255   # a white square on a black background

print('shape:', im.shape, 'dtype:', im.dtype)

plt.imshow(im, cmap='gray', vmin=0, vmax=255)
plt.title('Grayscale image (100x100)')
plt.axis('off')
plt.show()

## Các phép toán đơn giản trên điểm ảnh

Vì ảnh chỉ là một mảng số, các phép toán NumPy thông thường có thể áp dụng trực tiếp, ví dụ như đảo ngược cường độ.

In [ ]:
im_inv = 255 - im

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(im, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(im_inv, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Inverted (255 - pixel)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Ảnh màu là một mảng 3D

Kích thước của ảnh là `(height, width, channels)` &mdash; tức kết quả của `array.shape`. Trục **thứ nhất** chọn hàng (đi xuống dọc theo ảnh, tức $y$), trục **thứ hai** chọn cột (đi ngang qua ảnh, tức $x$); trục **thứ ba** chọn kênh màu (đỏ, lục hoặc lam). Theo quy ước, gốc tọa độ nằm ở góc trên bên trái.

**Cảnh báo:** Mặc dù `array.shape` và cách đánh chỉ số của NumPy dùng thứ tự `(row, column)` = `(y, x)`, nhiều hàm OpenCV nhận điểm hoặc kích thước &mdash; `cv2.circle`, `cv2.rectangle`, `cv2.resize`, tham số `dsize` của `cv2.warpAffine`, v.v. &mdash; lại yêu cầu thứ tự ngược lại, tức `(x, y)` hoặc `(width, height)`. Sự đảo ngược này là nguồn gốc phổ biến của lỗi; hãy luôn kiểm tra thứ tự tham số của hàm thay vì mặc định rằng nó khớp với shape của mảng.

In [ ]:
imc = np.zeros((50, 100, 3), dtype=np.uint8)
imc[:, :, 0] = 255        # red channel on for the whole image
imc[10:40, 20:80, 1] = 255  # add green in the middle -> red + green looks yellow

print('shape:', imc.shape, 'dtype:', imc.dtype)

plt.imshow(imc)
plt.title('Color image (100x100x3)')
plt.axis('off')
plt.show()

## Màu thường được sắp theo thứ tự RGB

Thông thường, thứ tự của 3 kênh màu là đỏ, lục và lam (RGB). Để minh họa, hãy tạo một ảnh nhỏ từ **sáu màu cơ bản tâm lý (psychological primaries)** &mdash; sắp xếp thành lưới 2 hàng, 3 cột. Đây là sáu sắc màu (đen, trắng, đỏ, vàng, lục, lam) mà hệ thị giác của con người xem là các màu nguyên thủy, không pha trộn.

In [ ]:
im_primaries = np.zeros((2, 3, 3), dtype=np.uint8)
im_primaries[0, 0, :] = (0, 0, 0)  # black
im_primaries[0, 1, :] = (255, 255, 255)  # white
im_primaries[0, 2, :] = (255, 0, 0)  # red
im_primaries[1, 0, :] = (255, 255, 0)  # yellow
im_primaries[1, 1, :] = (0, 255, 0)  # green
im_primaries[1, 2, :] = (0, 0, 255)  # blue
print('shape:', im_primaries.shape, '  (2 rows, 3 columns, 3 color channels)')

plt.imshow(im_primaries)
plt.title('The six psychological primaries')
plt.axis('off')
plt.show()

Cắt lát (slicing) theo trục **thứ nhất** sẽ chọn nguyên các hàng (một dải ngang của ảnh); cắt lát theo trục **thứ hai** sẽ chọn nguyên các cột (một dải dọc).

In [ ]:
top_row = im_primaries[0:1, :, :]        # first axis sliced -> a full-width horizontal band
left_column = im_primaries[:, 0:1, :]    # second axis sliced -> a full-height vertical band
# NOTE: a bare integer index (im_primaries[0, :, :]) would DROP that axis entirely, turning
# shape (1, 3, 3) into (3, 3) -- imshow would then misread it as a 3x3 grayscale image instead
# of a 1-row RGB strip. Slicing with 0:1 keeps the axis alive as size 1.

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(im_primaries)
axes[0].set_title('Full image', fontsize=9)
axes[1].imshow(top_row)
axes[1].set_title(f'Top row  [0:1, :, :]', fontsize=9)
axes[2].imshow(left_column)
axes[2].set_title(f'Left column  [:, 0:1, :]', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Đọc một ảnh thật: BGR và RGB

Tất cả các ảnh màu từ đầu đến giờ đều được tạo thủ công với kênh màu theo thứ tự RGB &mdash; khớp với những gì `plt.imshow` của Matplotlib mong đợi. Tuy nhiên, ảnh thật thường được đọc bằng `cv2.imread` của OpenCV, hàm này đọc (và ghi) ảnh màu với các kênh theo thứ tự **BGR** &mdash; lam trước, đỏ sau cùng &mdash; ngược với thứ tự RGB mà mọi thư viện xử lý ảnh/vẽ đồ thị Python khác đều giả định. Đưa thẳng một mảng BGR vào `plt.imshow` sẽ khiến các kênh màu bị hiểu sai, dẫn đến ảnh hiển thị kỳ lạ.

In [ ]:
im_bgr = cv2.imread('../img/rose.jpg')
print('shape:', im_bgr.shape, ' (last axis is in BGR order)')

plt.imshow(im_bgr)
plt.title('im_bgr shown directly -- wrong colors!')
plt.axis('off')
plt.show()

### Hai cách khắc phục vấn đề này

1. **Đảo ngược trục kênh màu bằng cách đánh chỉ số của NumPy.** `img[:, :, ::-1]` đảo ngược thứ tự của trục cuối cùng, biến `[B, G, R]` thành `[R, G, B]` &mdash; chỉ là một phép toán trên mảng, không cần hàm OpenCV nào.
2. **`cv2.cvtColor`.** Hàm chuyển đổi màu đa năng của OpenCV; `cv2.COLOR_BGR2RGB` thực hiện đúng phép hoán đổi kênh như trên, nhưng thể hiện rõ ràng hơn *lý do*, và là lựa chọn chuẩn mực trong code OpenCV (cũng hàm này còn xử lý những phép chuyển đổi không chỉ đơn giản là đảo ngược, ví dụ sang ảnh xám hoặc HSV, ở các bài sau).

Cả hai cách đều cho kết quả giống hệt nhau ở đây, vì chuyển đổi BGR&rarr;RGB chẳng qua chỉ là đảo ngược ba kênh.

In [ ]:
im_rgb1 = im_bgr[:, :, ::-1]
im_rgb2 = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)

print('the two methods agree exactly:', np.array_equal(im_rgb1, im_rgb2))

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, im, title in zip(axes, [im_bgr, im_rgb1, im_rgb2],
                          ['im_bgr (wrong)', 'img[:, :, ::-1]', "cv2.cvtColor(..., BGR2RGB)"]):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://picryl.com" target="_blank" rel="noopener">Picryl</a></p>

### Bài tập

1. Sửa ảnh màu `imc` để hình chữ nhật có màu lục lam (cyan) thay vì vàng. Bạn cần thay đổi (những) kênh nào?
2. Trong `im_primaries`, hãy cắt ra khối *dưới cùng bên phải* (màu lam) bằng các khoảng hàng và cột, rồi xác nhận bằng `np.array_equal` rằng nó trùng với một khối mới được tô màu `(0, 0, 255)`.
3. Đọc `rose.jpg` rồi in `im_bgr[0, 0, :]` (điểm ảnh trên cùng bên trái, thứ tự BGR) cạnh `im_rgb2[0, 0, :]` (thứ tự RGB). Tự kiểm tra rằng đó là cùng ba con số, chỉ khác thứ tự.